In [ ]:
# Setup (Colab & Local)
!pip install -r requirements.txt
import zipfile, os
if not os.path.exists('dataset'):
    print('Unzipping dataset...')
    with zipfile.ZipFile('dataset.zip', 'r') as z:
        z.extractall('.')
print('Environment setup complete.')


# Model List

## Classical Machine Learning Models
- Support Vector Machine (SVM – RBF)
- Random Forest (RF)
- Decision Tree (DT)

## Deep Learning Models (CNN Architectures)
- Inception V3
- ResNet50
- ResNet100
- ResNet152
- EfficientNet-B3
- EfficientNet-B5
- EfficientNet-B7
- VGG16
- VGG19
- MobileNet (V2/V3)

## Transformer Models
- Vision Transformer (ViT)
- Swin Transformer
- Cross-Attention Vision Transformer (CrossViT)

# %%
"""


Full notebook: Classical ML, Deep Learning, and Transformer models
on a brain tumour image classification dataset.

Outputs:
 - ROC curve (per-model)
 - Confusion matrix (per-model)
 - Optuna hyperparameter tuning for ML and DL

Assumptions:
 - You have a CSV `data/labels.csv` with columns: image_path,label
 - Images are in `data/images/` and paths in CSV are relative to that folder
 - Binary classification (label 0/1). For multi-class, minor edits required.

How to run:
 - Install requirements: pip install -r requirements.txt
   requirements.txt should include: scikit-learn, pandas, matplotlib, seaborn,
   opencv-python, torch, torchvision, timm, albumentations, optuna, xgboost, tqdm



In [ ]:
This is a single-file notebook (cells separated with # %%). Save as
`brain_tumour_full_notebook.py` and open in Jupyter (it will recognize cells),
or copy to a .ipynb if you prefer.
"""

# %%
# Basic imports
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, classification_report

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb

# Deep learning
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import optuna

# Repro
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.manual_seed(SEED)

# %%
# User-editable paths
DATA_CSV = 'dataset/data.csv'   # CSV with columns: image_path,label
IMAGE_ROOT = 'dataset'
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# %%
# Quick dataset check & load
df = pd.read_csv(DATA_CSV)
assert 'image_path' in df.columns and 'label' in df.columns, "CSV must have image_path and label"
print(f"Loaded {len(df)} rows. Classes: {df.label.value_counts().to_dict()}")

# %%
# Helper: create basic features for classical ML (simple: mean, std, histogram)
import cv2

def compute_basic_features(row, image_root=IMAGE_ROOT, bins=32):
    p = row['image_path']
    full = str(Path(image_root) / p) if not os.path.isabs(p) else p
    img = cv2.imread(full, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(full)
    img = cv2.resize(img, (128,128))
    feats = []
    feats.append(img.mean())
    feats.append(img.std())
    # histogram
    hist = cv2.calcHist([img],[0],None,[bins],[0,256]).flatten()
    hist = hist / (hist.sum()+1e-9)
    feats.extend(hist.tolist())
    return np.array(feats, dtype=np.float32)

# %%
# Build feature matrix (classical ML)
print('Computing basic image features (this may take a while)')
feat_list = []
for _,r in tqdm(df.iterrows(), total=len(df)):
    feat_list.append(compute_basic_features(r))
X_basic = np.vstack(feat_list)
y = df['label'].values

# %%
# Split for ML experiments
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X_basic, y, df.index.values, test_size=0.2, stratify=y, random_state=SEED)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

# %%
# Train some classical ML models with simple hyperparameter suggestions
ml_models = {
    'logreg': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'rf': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED),
    'svc': SVC(probability=True, class_weight='balanced', random_state=SEED),
    'xgb': xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED)
}

ml_results = {}
for name, m in ml_models.items():
    print('Training', name)
    m.fit(X_train_s, y_train)
    probs = m.predict_proba(X_test_s)
    preds = m.predict(X_test_s)
    try:
        roc = roc_auc_score(y_test, probs, multi_class='ovr')
    except:
        roc = 0.0
    #     # fpr, tpr, _ = roc_curve(y_test, probs)
    cm = confusion_matrix(y_test, preds)
    ml_results[name] = {'model':m, 'roc':roc, 'cm':cm}
    print(name, 'AUC', roc)

# %%
# Plot classical ML ROC curves
plt.figure(figsize=(8,6))
for name,res in ml_results.items():
    #     plt.plot(res['fpr'], res['tpr'], label=f"{name} (AUC={res['roc']:.3f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Classical ML ROC')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(OUT_DIR,'ml_roc.png'))
plt.show()

# %%
# Show confusion matrices
for name,res in ml_results.items():
    plt.figure(figsize=(4,3))
    sns.heatmap(res['cm'], annot=True, fmt='d', cmap='Blues')
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Pred')
    plt.ylabel('True')
    plt.savefig(os.path.join(OUT_DIR,f'ml_cm_{name}.png'))
    plt.show()

# %%
# Now build PyTorch dataset for DL & Transformer
class BrainTumorDataset(Dataset):
    def __init__(self, df, image_root, transforms=None):
        self.df = df.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = row['image_path']
        full = str(self.image_root / p) if not os.path.isabs(p) else p
        img = cv2.imread(full)
        if img is None:
            raise FileNotFoundError(full)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transforms:
            img = self.transforms(image=img)['image']
        label = int(row['label'])
        return img, label

# %%
# Transforms
def get_transforms(img_size=224, is_train=True):
    if is_train:
        return A.Compose([
            A.Resize(img_size,img_size),
            A.RandomRotate90(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
            A.VerticalFlip(p=0.2),
            A.Normalize(),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size,img_size),
            A.Normalize(),
            ToTensorV2(),
        ])

# %%
# Simple training utilities
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    losses = []
    preds = []
    targets = []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        preds.extend(torch.argmax(out.detach(), dim=1).cpu().numpy().tolist())
        targets.extend(labels.cpu().numpy().tolist())
    acc = (np.array(preds)==np.array(targets)).mean()
    return np.mean(losses), acc

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    losses=[]
    preds=[]
    probs=[]
    targets=[]
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        out = model(imgs)
        loss = criterion(out, labels)
        losses.append(loss.item())
        prob = torch.softmax(out, dim=1).cpu().numpy()
        p = torch.argmax(out, dim=1).cpu().numpy()
        probs.extend(prob.tolist())
        preds.extend(p.tolist())
        targets.extend(labels.cpu().numpy().tolist())
    acc = (np.array(preds)==np.array(targets)).mean()
    try:
        roc = roc_auc_score(targets, probs, multi_class='ovr')
    except Exception:
        roc = float('nan')
    return np.mean(losses), acc, roc, probs, preds, targets

# %%
# Model factories: CNN and ViT from timm
def create_cnn(num_classes=4, model_name='resnet18', pretrained=True, dropout=0.2):
    m = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
    return m

# %%
# Quick PyTorch training run function
def run_pytorch_training(df, model_name='resnet18', img_size=224, bs=32, epochs=10, lr=1e-4):
    # split
    # Stratified Block Split (to prevent data leakage)
    train_dfs = []
    val_dfs = []
    for label in df.label.unique():
        sub = df[df.label==label].sort_values('image_path')
        split_idx = int(len(sub)*0.8)
        train_dfs.append(sub.iloc[:split_idx])
        val_dfs.append(sub.iloc[split_idx:])
    train_df = pd.concat(train_dfs).reset_index(drop=True)
    val_df = pd.concat(val_dfs).reset_index(drop=True)
    train_ds = BrainTumorDataset(train_df, IMAGE_ROOT, transforms=get_transforms(img_size, is_train=True))
    val_ds = BrainTumorDataset(val_df, IMAGE_ROOT, transforms=get_transforms(img_size, is_train=False))

    # Class balancing
    class_counts = train_df.label.value_counts().sort_index().values
    class_weights = 1. / (class_counts + 1e-6)
    sample_weights = [class_weights[l] for l in train_df.label]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    train_loader = DataLoader(train_ds, batch_size=bs, sampler=sampler, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=4)

    model = create_cnn(num_classes=4, model_name=model_name).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2, verbose=True)
    early_stopping_patience = 5
    epochs_no_improve = 0

    best_roc = 0
    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[], 'val_roc':[]}
    for e in range(1, epochs+1):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc, val_roc, probs, preds, targets = validate(model, val_loader, criterion, DEVICE)
        scheduler.step(val_roc)
        print(f"Epoch {e}: tr_loss {tr_loss:.4f} tr_acc {tr_acc:.4f} | val_loss {val_loss:.4f} val_acc {val_acc:.4f} val_roc {val_roc:.4f}")
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss); history['val_acc'].append(val_acc); history['val_roc'].append(val_roc)
        if val_roc > best_roc:
            epochs_no_improve = 0
            best_roc = val_roc
            torch.save({'model_state': model.state_dict(), 'model_name':model_name}, os.path.join(OUT_DIR,f'best_{model_name}.pth'))
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= early_stopping_patience:
                print(f'Early stopping at epoch {e}!')
                break
    return model, history, probs, preds, targets

# %%
# Run a small example (ResNet18)
model_resnet, hist_resnet, resnet_probs, resnet_preds, resnet_targets = run_pytorch_training(df, model_name='resnet18', img_size=224, bs=32, epochs=6, lr=2e-4)

# %%
# Plot ROC for DL model
    # fpr, tpr, _ = roc_curve(resnet_targets, resnet_probs)
    # roc_auc = auc(fpr,tpr)
plt.figure(figsize=(6,5))
    # plt.plot(fpr,tpr,label=f"resnet (AUC={roc_auc:.3f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('DL ROC')
plt.legend(); plt.grid(True);
plt.savefig(os.path.join(OUT_DIR,'dl_resnet_roc.png'))
plt.show()

# Confusion matrix
cm = confusion_matrix(resnet_targets, resnet_preds)
plt.figure(figsize=(4,3)); sns.heatmap(cm, annot=True, fmt='d'); plt.title('ResNet Confusion Matrix'); plt.show()

# %%
# Optional: Vision Transformer run (fine-tune)
model_vit, hist_vit, vit_probs, vit_preds, vit_targets = run_pytorch_training(df, model_name='vit_tiny_patch16_224', img_size=224, bs=32, epochs=6, lr=5e-5)

# %%
# Plot both DL ROC curves together
    # fpr1, tpr1, _ = roc_curve(resnet_targets, resnet_probs)
    # fpr2, tpr2, _ = roc_curve(vit_targets, vit_probs)
plt.figure(figsize=(7,6))
    # plt.plot(fpr1,tpr1,label=f"ResNet (AUC={auc(fpr1,tpr1):.3f})")
    # plt.plot(fpr2,tpr2,label=f"ViT (AUC={auc(fpr2,tpr2):.3f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('DL Model Comparison ROC')
plt.legend(); plt.grid(True); plt.savefig(os.path.join(OUT_DIR,'dl_comparison_roc.png'))
plt.show()

# %%
# Hyperparameter tuning suggestions
print('Suggestions:')
print('- For classical ML: use GridSearchCV or Optuna to tune n_estimators, max_depth (RF), C (logreg/SVM), learning_rate and n_estimators for XGBoost')
print('- For DL: tune learning rate, weight_decay, batch_size, img_size, model_name, dropout, augmentation strength, scheduler')
print('- Use Optuna for automated search (example below)')

# %%
# Quick Optuna example for tuning a light PyTorch model (search lr, batch_size, model_name)
def optuna_objective(trial):
    model_name = trial.suggest_categorical('model_name', ['resnet18', 'efficientnet_b0', 'mobilenetv3_small'])
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-3)
    bs = trial.suggest_categorical('bs', [8,16,32])
    epochs = 6
    try:
        model, hist, probs, preds, targets = run_pytorch_training(df, model_name=model_name, img_size=224, bs=bs, epochs=epochs, lr=lr)
    except Exception as e:
        print('trial failed', e)
        return 0.0
        roc = roc_auc_score(targets, probs, multi_class='ovr')
    return roc

# Run a quick study (small n_trials for demo)
study = optuna.create_study(direction='maximize')
study.optimize(optuna_objective, n_trials=6)
print('Best params:', study.best_params)

# %%
# Save study results
study.trials_dataframe().to_csv(os.path.join(OUT_DIR,'optuna_trials.csv'), index=False)

# %%
# Wrap-up: show top results
print('Classical ML AUCs:')
for k,v in ml_results.items():
    print(k, v['roc'])
print('DL ResNet AUC:', roc_auc_score(resnet_targets, resnet_probs, multi_class='ovr'))
print('DL ViT AUC:', roc_auc_score(vit_targets, vit_probs, multi_class='ovr'))

# Save final figures and models already saved during training in OUT_DIR
print('Outputs saved to', OUT_DIR)
